## CASE STUDY 1

In [41]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

Wczytanie danych

In [42]:
df = pd.read_csv('/content/drive/MyDrive/waga zajecia/creditcard.csv')
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [43]:
class_names = {0:'Not Fraud', 1:'Fraud'}
print(df.Class.value_counts().rename(index = class_names))

df.isnull().sum().sum()

Class
Not Fraud    284315
Fraud           492
Name: count, dtype: int64


np.int64(0)

Standaryzacja danych

In [44]:
scaler = StandardScaler()
df[['Time', 'Amount']] = scaler.fit_transform(df[['Time', 'Amount']])

#Podział na X i y
X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

PCA

In [45]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

pca = PCA().fit(X_train_scaled)

var_cum = np.cumsum(pca.explained_variance_ratio_)
var90 = np.searchsorted(var_cum, 0.90) + 1

print(f'Liczba komponentów wymagana do wyjaśnienia 90% wariancji: {var90}')


Liczba komponentów wymagana do wyjaśnienia 90% wariancji: 26


In [46]:
pca_final = PCA(n_components=26, random_state=42)
X_train_pca = pca_final.fit_transform(X_train_scaled)
X_test_pca = pca_final.transform(scaler.transform(X_test))

### Model wrażliwy na koszty

In [47]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

model_lr = LogisticRegression(solver='liblinear', class_weight='balanced', random_state=42)
model_lr.fit(X_train_pca, y_train)

print("Model 1: Regresja Logistyczna (Cost-Sensitive)")
y_pred_lr = model_lr.predict(X_test_pca)
print(classification_report(y_test, y_pred_lr))
print(confusion_matrix(y_test, y_pred_lr))

Model 1: Regresja Logistyczna (Cost-Sensitive)
              precision    recall  f1-score   support

           0       1.00      0.98      0.99     85295
           1       0.06      0.89      0.12       148

    accuracy                           0.98     85443
   macro avg       0.53      0.93      0.55     85443
weighted avg       1.00      0.98      0.99     85443

[[83313  1982]
 [   17   131]]


Model poprawnie identyfikuje 89% wszystkich faktycznych oszustw. Jednakże, precyzja jest na poziomie 0.06, co wskazuje, że 94% jego alarmów to błędnie wskazane legalne transakcje. Niski F1-score (0.12) dowodzi, że model jest niezrównoważony, gdyż jego zdolność do wykrywania jest całkowicie zniwelowana przez ogromną liczbę generowanych pomyłek.

###Model Las Losowy

In [48]:
from sklearn.ensemble import RandomForestClassifier
from imblearn.pipeline import make_pipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

over = SMOTE(sampling_strategy=0.1, random_state=42)
under = RandomUnderSampler(sampling_strategy=0.5, random_state=42)


model_rf = RandomForestClassifier(random_state=42)
pipeline_rf = make_pipeline(over, under, model_rf)
pipeline_rf.fit(X_train_pca, y_train)

print("Model 2: Las Losowy (SMOTE i RUS)")
y_pred_rf = pipeline_rf.predict(X_test_pca)
print(classification_report(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))

Model 2: Las Losowy (SMOTE i RUS)
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     85295
           1       0.64      0.80      0.71       148

    accuracy                           1.00     85443
   macro avg       0.82      0.90      0.86     85443
weighted avg       1.00      1.00      1.00     85443

[[85228    67]
 [   29   119]]


Model Lasu Losowego poprawnie identyfikuje 82% wszystkich faktycznych transakcji oszukańczych (wykrywając 121 ze 148). Model charakteryzuje się znacznie lepszą precyzją wynoszącą 0.66. Wskazuje to, że 66% wszystkich transakcji oznaczonych przez model jako oszustwa faktycznie nimi było, a liczba fałszywych alarmów spadła.
Wynik F1-score (0.73) powtierdza, że model ten jest bardziej zrównoważony.

###Hiperparametryzacja Lasu Losowego


In [53]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, average_precision_score

rf = RandomForestClassifier(random_state=42)

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': ['balanced', 'balanced_subsample', None]
}

cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

In [59]:
print("Rozpoczynam RandomizedSearchCV...")
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_grid,
    n_iter=10,
    cv=cv_strategy,
    scoring='average_precision',
    refit='average_precision',
    n_jobs=-1,
    random_state=42,
    verbose=2
)

random_search.fit(X_train_pca, y_train)

print("Strojenie zakończone.")

Rozpoczynam RandomizedSearchCV...
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Strojenie zakończone.


In [65]:
print(f"Najlepsze parametry znalezione przez RandomSearch: {random_search.best_params_}")
print(f"Najlepszy wynik (AUC-PR) na CV: {random_search.best_score_:.4f}")
best_rf_model = random_search.best_estimator_

Najlepsze parametry znalezione przez RandomSearch: {'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': 10, 'class_weight': None}
Najlepszy wynik (AUC-PR) na CV: 0.8476


In [64]:
y_pred_best_rf = best_rf_model.predict(X_test_pca)
y_proba_best_rf = best_rf_model.predict_proba(X_test_pca)[:, 1]

print(classification_report(y_test, y_pred_best_rf))
print(confusion_matrix(y_test, y_pred_best_rf))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     85295
           1       0.96      0.73      0.83       148

    accuracy                           1.00     85443
   macro avg       0.98      0.86      0.91     85443
weighted avg       1.00      1.00      1.00     85443

[[85290     5]
 [   40   108]]


Ten model cechuje wysoka prezycja. Tylko 5 fałszywych alarmów, natomiast ma słabszą wykrywalność. Ominął 40 oszustw. F1-score jest wysoki, wyraźnie faworyzując pewność kosztem pełnej wykrywalności.

###Podsumowanie



### Podsumowanie i wnioski biznesowe

Porównanie trzech podejść wykazało różnice w zachowaniu modeli, co pozwala na dobranie strategii w zależności od kosztów biznesowych przypisanych do błędów typu *False Positive* (fałszywy alarm) oraz *False Negative* (przepuszczone oszustwo).

#### Zestawienie wyników na zbiorze testowym:

| Model | Precision | Recall | F1-Score | Fałszywe alarmy (FP) | Przepuszczone fraudy (FN) |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **Model 1:** Regresja Logistyczna | 0.06 | **0.89** | 0.12 | 1982 | **17** |
| **Model 2:** Las Losowy (SMOTE+RUS) | 0.64 | 0.80 | 0.71 | 67 | 29 |
| **Model 3:** Strojony Las Losowy | **0.96** | 0.73 | **0.83** | **5** | 40 |

#### Wnioski:

* **Model 1 (Regresja Logistyczna):** Jest **nieużyteczny operacyjnie**. Mimo wysokiej wykrywalności (Recall 0.89), generuje aż 1982 fałszywe alarmy. W praktyce oznaczałoby to zablokowanie kart prawie dwóm tysiącom uczciwych klientów, paraliżując dział obsługi i niszcząc UX.
* **Model 2 (Las Losowy z SMOTE+RUS):** Stanowi **agresywne podejście** nastawione na maksymalizację ochrony kapitału. Przepuszcza tylko 29 oszustw (o 11 mindre niż Model 3), ale płaci za to 67 fałszywymi blokadami.
* **Model 3 (Strojony Las Losowy):** Okazał się **najlepiej zrównoważony**, co potwierdza najwyższy wskaźnik F1-score (0.83). Z punktu widzenia stabilności biznesowej i kosztów operacyjnych jest to wariant optymalny – redukuje liczbę fałszywych alarmów do zaledwie 5, zachowując przy tym akceptowalną wykrywalność.